In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [2]:
# Q6: PHÁT HIỆN BẤT THƯỜNG ÁP SUẤT THẤP KHI MÁY ĐANG VẬN HÀNH
q6_anomalies = spark.sql('''
    SELECT
        date,
        COUNT(*) AS so_lan_ap_thap
    FROM sensor
    WHERE COMP = 0
      AND Reservoirs < 8.0
    GROUP BY date
    ORDER BY so_lan_ap_thap DESC
    LIMIT 10
''')
print("\n========== EXECUTION PLAN ==========")
q6_anomalies.explain(True)
print("--- TOP 10 NGÀY CÓ ÁP SUẤT BÌNH CHỨA < 8 BAR ---")
df_q6_anomalies = q6_anomalies.toPandas()
try:
    display(df_q6_anomalies)
except NameError:
    print(df_q6_anomalies.to_string(index=False))


========== EXECUTION PLAN ==========
== Parsed Logical Plan ==
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['so_lan_ap_thap DESC NULLS LAST], true
      +- 'Aggregate ['date], ['date, 'COUNT(1) AS so_lan_ap_thap#711]
         +- 'Filter (('COMP = 0) AND ('Reservoirs < 8.0))
            +- 'UnresolvedRelation [sensor], [], false

== Analyzed Logical Plan ==
date: date, so_lan_ap_thap: bigint
GlobalLimit 10
+- LocalLimit 10
   +- Sort [so_lan_ap_thap#711L DESC NULLS LAST], true
      +- Aggregate [date#24], [date#24, count(1) AS so_lan_ap_thap#711L]
         +- Filter ((COMP#9 = 0) AND (Reservoirs#6 < cast(8.0 as double)))
            +- SubqueryAlias sensor
               +- View (`sensor`, [_c0#0, timestamp#1, TP2#2, TP3#3, H1#4, DV_pressure#5, Reservoirs#6, Oil_temperature#7, Motor_current#8, COMP#9, DV_electric#10, Towers#11, MPG#12, LPS#13, Pressure_switch#14, Oil_level#15, Caudal_impulses#16, hour#21, month#22, dow#19, weekday#23, date#24])
                  +- Project [_c0#0, t

,date,so_lan_ap_thap
0,2020-05-20,5821
1,2020-06-07,4861
2,2020-05-13,3503
3,2020-06-06,1652
4,2020-06-05,932
5,2020-07-15,758
6,2020-05-19,475
7,2020-07-17,451
8,2020-05-14,385
9,2020-06-08,380
